# FUNGI v9.0 — Pruning Pipeline
**Functional Unravelling of Network Geometry for Inference**

## Setup

In [1]:
import os, yaml, warnings, sys, gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*DataFrame is highly fragmented.*')

CONFIG_PATH = Path('fungi_config.yaml')
with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

if cfg['runtime'].get('single_threaded_blas', True):
    for var in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
        os.environ[var] = '1'

RAW_GRAPH_PATH = Path(cfg['input']['graph_path'])
SC_DATA_PATH   = Path(cfg['input']['sc_data_path'])
OUTPUT_ROOT    = Path(cfg['output']['root_dir'])
SRC_ROOT       = Path('src')

for phase in cfg['output']['phases']:
    (OUTPUT_ROOT / phase).mkdir(parents=True, exist_ok=True)
Path(cfg['output']['figures_dir']).mkdir(parents=True, exist_ok=True)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f'Configuration loaded from {CONFIG_PATH}')

Configuration loaded from fungi_config.yaml


## Phase 1: Data Ingestion

In [2]:
from graph_utils import load_graph

raw_G, raw_sparse_mat = load_graph(RAW_GRAPH_PATH)
N_GENES = raw_sparse_mat.shape[0]
print(f'Parent graph: {N_GENES:,} nodes, {raw_sparse_mat.nnz:,} edges')

adata = sc.read_h5ad(str(SC_DATA_PATH))
print(f'Expression data: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes')

Loading graph from VCC_chitin_parent_graph.parquet ...
  Detected Parquet format.
  Columns detected: source='Regulator', target='Target', weight='Importance'
  Renamed 'Importance' -> 'weight' for NetworkX compatibility.
  Weight range in sparse matrix: [0.0000, 2921.6518]
  Nodes: 5,024
  Edges: 25,235,546
  Density: 100.0000%
Parent graph: 5,024 nodes, 25,235,546 edges
Expression data: 4,117 cells x 5,024 genes


## Phase 0: Diagnostic Calibration

In [3]:
from diagnostics import run_diagnostics

utopian_bounds, loss_weights, diagnostic_report = run_diagnostics(
    adata=adata, n_genes=N_GENES,
    cfg_diagnostics=cfg['diagnostics'],
    cfg_input=cfg['input'],
    raw_sparse_mat=raw_sparse_mat,
)

lam_eff = diagnostic_report['lam_eff']

diag_dir = OUTPUT_ROOT / 'phase0_diagnostics'
report_clean = {k: v for k, v in diagnostic_report.items() if not k.startswith('_')}
pd.DataFrame([report_clean]).to_json(diag_dir / 'diagnostic_report.json', indent=2)
print('Diagnostic report saved.')

Phase 0: Building perturbation impact array...
  Metacell pooling (factor=10): LFC cutoff 0.250 → 0.079
  96 perturbation groups detected.


  LFC proxy: 100%|██████████████████████████| 96/96 [00:00<00:00, 3989.19pert/s]

  Running Wilcoxon on all 96 perturbations.



  DEG counts: 100%|███████████████████████████| 96/96 [00:02<00:00, 37.63pert/s]


  Active perts    : 96 / 96 tested
  DEG matrix      : 54,516 causal edges
  LFC matrix      : 96 perturbations × 5024 genes

  λ_center = 20.00 edges/gene

  Running probes...

FUNGI v8.0 — Phase 0 Diagnostic Summary
  Dataset: 5,024 HVGs | 96/96 active perturbations
  Estimated graph density (λ_center): 20.0 edges/gene

  ~ alpha [1.6047, 1.9967]  conf=0.10  wt=3.0  — Scale-free degree exponent
  ✓  gini [0.5519, 0.7519]  conf=0.65  wt=19.2  — Regulatory inequality (hub dominance)
  ✓ S_max [0.0482, 0.0782]  conf=0.70  wt=20.7  — Largest hub's target fraction
  ~     Q [0.3000, 0.7000]  conf=0.33  wt=9.9  — Functional module separation
  ✓     C [0.0534, 0.1134]  conf=0.70  wt=20.7  — Feed-forward loop density
  ✓   rho [-0.2159, -0.0559]  conf=0.90  wt=26.6  — Hub-to-effector disassortativity

  Decision: PROCEED

Diagnostic report saved.


## Phase 2: Graph Normalization + Pre-computation

In [4]:
from scipy.stats import rankdata
from filtering import adaptive_threshold_filter
from engine import (compute_source_quantile_weights,
                    compute_pagerank_kappa_multipliers,
                    compute_source_pert_impact)

G_work = adaptive_threshold_filter(
    raw_sparse_mat, target_density=cfg['prefilter']['target_density'])
gc.collect()

G_work_coo = G_work.tocoo()
sources_raw = G_work_coo.row.copy()
targets_raw = G_work_coo.col.copy()
weights_raw = G_work_coo.data.copy()

W_ranked = rankdata(weights_raw, method='average').astype(np.float64) / len(weights_raw)
if W_ranked.max() - W_ranked.min() < 1e-6:
    print('WARNING: Degenerate weights — log1p fallback')
    W_log = np.log1p(weights_raw.astype(np.float64))
    W_arr = (np.clip(W_log / max(W_log.max(), 1e-10), 0.001, 1.0)
             if W_log.max() > 1e-12
             else np.random.default_rng(42).uniform(0.01, 1.0, len(weights_raw)))
else:
    W_arr = W_ranked

out_deg_raw = np.bincount(sources_raw, minlength=N_GENES).astype(np.float64)
D_arr       = np.log1p(out_deg_raw)[sources_raw]
sources_arr = sources_raw
targets_arr = targets_raw

print(f'Candidate pool: {len(W_arr):,} edges')

# Source-quantile weights
W_source_quantile = compute_source_quantile_weights(sources_arr, W_arr, N_GENES)

# PageRank kappa multipliers
pr_cfg = cfg.get('pagerank_kappa', {})
per_gene_kappa = compute_pagerank_kappa_multipliers(
    G_work, N_GENES,
    alpha=pr_cfg.get('alpha', 0.85),
    n_iter=pr_cfg.get('n_iter', 60),
    hub_percentile=pr_cfg.get('hub_percentile', 99.0),
    hub_multiplier=pr_cfg.get('hub_multiplier', 3.0),
)
n_hubs = int((per_gene_kappa > 1.5).sum())
print(f'  {n_hubs} hub genes identified via PageRank')

# Perturbation impact prior
impact_arr_raw = np.array(diagnostic_report.get('_impact_array', []), dtype=np.float64)
pert_labels    = np.array(diagnostic_report.get('_perturbation_labels', []))
name_to_idx    = diagnostic_report.get('_name_to_idx', {})
source_pert_impact = compute_source_pert_impact(
    impact_arr_raw, pert_labels, name_to_idx, N_GENES)
print(f'  {int((source_pert_impact != 1.0).sum())} genes with non-neutral impact prior')

Filtering graph to 10.0% density...
  Filtered to 2,524,057 edges (10.0000% density).
Candidate pool: 2,524,057 edges
  51 hub genes identified via PageRank
  96 genes with non-neutral impact prior


## Phase 3: Expansive Sobol Search

In [5]:
from search import generate_sobol_samples, SearchEvaluator

es_cfg = cfg['expansive_search']
hp_cfg = cfg['hyperparameter_bounds']

sobol_params, lower_bounds, upper_bounds = generate_sobol_samples(
    n_genes=N_GENES, n_samples=es_cfg['n_samples'],
    hp_cfg=hp_cfg, seed=es_cfg['random_seed'], lam_eff=lam_eff)

pert_col   = cfg['input']['perturbation_column']
ctrl_label = cfg['input']['control_label']
pert_genes = [g for g in adata.obs[pert_col].unique() if g != ctrl_label]
gene_list  = list(adata.var_names)
perturbed_nodes = np.array(
    [gene_list.index(g) for g in pert_genes
     if g in gene_list and gene_list.index(g) < N_GENES], dtype=int)
print(f'  Perturbation targets: {len(perturbed_nodes):,} genes')

evaluator = SearchEvaluator(
    W_arr=W_arr, W_q_arr=W_source_quantile, D_arr=D_arr,
    sources_arr=sources_arr, targets_arr=targets_arr,
    n_genes=N_GENES, perturbed_nodes=perturbed_nodes,
    utopian_bounds=utopian_bounds, loss_weights=loss_weights,
    shatter_cfg=cfg['shatter'],
    per_gene_kappa=per_gene_kappa,
    source_pert_impact=source_pert_impact,
    n_workers=es_cfg['n_workers'])

t0 = time.time()
shard_dir = str(OUTPUT_ROOT / 'phase3_expansive_search' / 'shards')
df_expansive = evaluator.evaluate(
    param_list=sobol_params, chunk_size=es_cfg['chunk_size'],
    shard_dir=shard_dir, desc='Phase 3: Sobol Search')

elapsed = time.time() - t0
n_viable    = int((df_expansive['is_shattered'] == 0).sum())
n_shattered = len(df_expansive) - n_viable
best_loss   = df_expansive.loc[df_expansive['is_shattered'] == 0, 'utopia_loss'].min() if n_viable > 0 else float('inf')
n_zero      = int((df_expansive['utopia_loss'] <= 1e-6).sum())

print(f'\nPhase 3: {len(df_expansive):,} graphs in {elapsed:.1f}s')
print(f'  Viable: {n_viable:,} | Shattered: {n_shattered:,}')
print(f'  Zero-loss: {n_zero:,} | Best loss: {best_loss:.6f}')
df_expansive.to_csv(OUTPUT_ROOT / 'phase3_expansive_search' / 'expansive_results.csv', index=False)

  Sobol sequence: 4,096 points in 6D space.
  Parameter space: β[1.0,3.0]  δ[0.00,1.00]  κ[0.020,0.150]  k_core[8,22]  λ[8.0,30.0]  ψ[0.0,2.0]
  Perturbation targets: 96 genes


Phase 3: Sobol Search:   0%|          | 0/4096 [00:00<?, ?graph/s]


Phase 3: 4,096 graphs in 1575.9s
  Viable: 4,072 | Shattered: 24
  Zero-loss: 9 | Best loss: 0.000000


## Phase 4: Spatial Niching

In [6]:
from niching import extract_anchors

if n_viable > 0:
    anchor_coords, anchor_losses, cluster_summary = extract_anchors(
        df_results=df_expansive,
        top_fraction=cfg['niching']['top_fraction'],
        n_clusters=cfg['niching']['n_clusters'],
        random_seed=cfg['runtime']['random_seed'])
    phase4_dir = OUTPUT_ROOT / 'phase4_niching'
    np.save(phase4_dir / 'anchor_coords.npy', anchor_coords)
    cluster_summary.to_csv(phase4_dir / 'cluster_summary.csv', index=False)
else:
    print('ERROR: No viable Phase 3 graphs.')

Spatial Niching: 203 elite graphs selected (top 5.0% of 4,072 survivors).
  Extracted 10 anchor coordinates across 10 spatial niches.
  Loss range of anchors: [0.0000, 0.4723]


## Phase 5: ML+GMM Refinement

In [7]:
from refinement import run_ml_gmm_refinement

df_refinement = None
if n_viable > 0:
    df_refinement, was_skipped = run_ml_gmm_refinement(
        df_phase3=df_expansive, lower=lower_bounds, upper=upper_bounds,
        evaluator=evaluator, refinement_cfg=cfg['refinement'])
    if df_refinement is not None:
        df_refinement.to_csv(
            OUTPUT_ROOT / 'phase5_refinement' / 'refinement_results.csv', index=False)

  Phase 5 SKIPPED: 9 zero-loss solutions found in Phase 3 (best=0.000000)


## Phase 6: Champion + Diverse Cohort Selection

Selects the champion (best loss) plus 4 topologically diverse alternates.
Alternates are chosen by farthest-point sampling in normalized topology space,
ensuring each has a meaningfully different topology — not just a trivially
different β or κ.

In [8]:
from refinement import select_champion, select_diverse_cohort

frames = [df_expansive]
if df_refinement is not None:
    frames.append(df_refinement)
df_all = pd.concat(frames, ignore_index=True)

# Build cohort of 5 topologically diverse candidates
cohort = select_diverse_cohort(df_all, utopian_bounds, N_GENES, cohort_size=5)

# Display cohort matrix
topo_cols = ['alpha', 'Gini', 'S_max', 'Q', 'C', 'rho']
hp_cols   = ['beta', 'delta', 'kappa', 'k_core', 'lambda', 'psi']

print('=' * 90)
print('FUNGI v9.0 — Candidate Cohort (5 topologically diverse graphs)')
print('=' * 90)
print()

# Header
print(f'{"#":>3s} {"Loss":>8s} │ {"α":>6s} {"Gini":>6s} {"S_max":>6s} '
      f'{"Q":>6s} {"C":>6s} {"ρ":>7s} │ '
      f'{"β":>5s} {"δ":>5s} {"κ":>5s} {"k_c":>5s} {"λ":>6s} {"ψ":>5s} │ {"Edges":>7s}')
print('─' * 90)

for _, row in cohort.iterrows():
    rank = int(row['cohort_rank'])
    star = ' ★' if row['is_champion'] else '  '
    
    # Check each topo param against bounds
    topo_status = []
    for param, col in [('alpha','alpha'),('gini','Gini'),('S_max','S_max'),
                       ('Q','Q'),('C','C'),('rho','rho')]:
        lo, hi = utopian_bounds[param]
        val = row[col]
        topo_status.append(f'{val:6.3f}')
    
    print(f'{rank:>2d}{star} {row["utopia_loss"]:8.4f} │ '
          f'{" ".join(topo_status)} │ '
          f'{row["beta"]:5.2f} {row["delta"]:5.2f} {row["kappa"]:5.3f} '
          f'{row["k_core"]:5.1f} {row["lambda"]:6.2f} {row["psi"]:5.2f} │ '
          f'{int(row["n_edges"]):>7,d}')

print('─' * 90)
print('★ = Champion (lowest utopia loss)')
print()

# Show bound compliance for champion
champ = cohort[cohort['is_champion']].iloc[0]
print('Champion topology vs targets:')
for param, col in [('alpha','alpha'),('gini','Gini'),('S_max','S_max'),
                   ('Q','Q'),('C','C'),('rho','rho')]:
    lo, hi = utopian_bounds[param]
    val = champ[col]
    inside = '✓' if lo <= val <= hi else '✗'
    print(f'  {inside} {param:>5s} = {val:.4f}  (target: [{lo:.4f}, {hi:.4f}])')

FUNGI v9.0 — Candidate Cohort (5 topologically diverse graphs)

  #     Loss │      α   Gini  S_max      Q      C       ρ │     β     δ     κ   k_c      λ     ψ │   Edges
──────────────────────────────────────────────────────────────────────────────────────────
 1 ★   0.0000 │  1.672  0.571  0.053  0.364  0.087 -0.086 │  1.13  0.80 0.148  19.4  12.09  0.35 │  60,741
 2     1.6226 │  2.043  0.633  0.110  0.384  0.061 -0.099 │  1.40  0.78 0.114  10.1   8.04  1.84 │  40,418
 3     1.7224 │  1.479  0.369  0.097  0.285  0.054 -0.071 │  1.78  0.99 0.149  12.3  15.52  1.89 │  77,963
 4     0.5842 │  1.519  0.728  0.091  0.399  0.080 -0.099 │  1.06  0.75 0.111  14.8   9.00  1.08 │  45,239
 5     1.7450 │  1.456  0.332  0.048  0.299  0.056 -0.073 │  2.28  0.81 0.143  20.1  16.32  0.59 │  81,975
──────────────────────────────────────────────────────────────────────────────────────────
★ = Champion (lowest utopia loss)

Champion topology vs targets:
  ✓ alpha = 1.6715  (target: [1.6047, 1.9967])


## Phase 7: Graph Output

Select which graphs to output as parquet files for SPECTRA.

**Set `graphs_to_output` below.** Default: `[1]` (champion only).  
To also output alternates: `[1, 3, 5]` or `[1, 2, 3, 4, 5]`.

In [9]:
# ═══════════════════════════════════════════════════════
# USER SELECTION: which graphs to output
# Default: [1] = champion only
# To output alternates: [1, 3] or [1, 2, 3, 4, 5]
graphs_to_output = [1]
# ═══════════════════════════════════════════════════════

In [10]:
from engine import build_graph_from_params

gene_names = list(adata.var_names)
output_dir = OUTPUT_ROOT / 'phase6_champion'

for rank_num in graphs_to_output:
    row = cohort[cohort['cohort_rank'] == rank_num]
    if len(row) == 0:
        print(f'  WARNING: No graph with cohort_rank={rank_num}')
        continue
    row = row.iloc[0]
    
    params = np.array([
        row['beta'], row['delta'], row['kappa'],
        row['k_core'], row['lambda'], row['psi']
    ])
    
    # Reconstruct exact graph from hyperparameters
    surv_s, surv_t, surv_W = build_graph_from_params(
        params, evaluator.Ws, evaluator.Wqs, evaluator.Ds,
        evaluator.srcs, evaluator.tgts,
        N_GENES, perturbed_nodes, cfg['shatter'],
        per_gene_kappa, source_pert_impact)
    
    # Build edge list with gene names
    graph_df = pd.DataFrame({
        'Regulator': [gene_names[s] for s in surv_s],
        'Target':    [gene_names[t] for t in surv_t],
        'Weight':    surv_W,
    })
    
    tag = 'champion' if row['is_champion'] else f'alternate_{rank_num}'
    out_path = output_dir / f'fungi_{tag}.parquet'
    graph_df.to_parquet(out_path, index=False)
    
    print(f'  [{tag}] {len(graph_df):,} edges → {out_path.name}')
    print(f'    loss={row["utopia_loss"]:.4f}  λ={row["lambda"]:.2f}  '
          f'β={row["beta"]:.3f}  δ={row["delta"]:.3f}  ψ={row["psi"]:.3f}')

# Save cohort recipe CSV (always)
cohort.to_csv(output_dir / 'cohort_recipes.csv', index=False)

# Save pipeline artifacts
import joblib as jl
jl.dump({
    'N_GENES': N_GENES,
    'utopian_bounds': utopian_bounds,
    'loss_weights': loss_weights,
    'diagnostic_report': report_clean,
    'champion': cohort[cohort['is_champion']].iloc[0].to_dict(),
}, OUTPUT_ROOT / 'pipeline_artifacts.joblib')

print('\nFUNGI v9.0 pipeline complete.')

ImportError: cannot import name 'build_graph_from_params' from 'engine' (/scratch/patrick.sheehan/FUNGI+/src/engine.py)

## Cleanup (Optional)

Deletes all intermediate data (shards, CSVs, diagnostics) so the next
run starts fresh. Keeps only the final outputs in `phase6_champion/`
and `pipeline_artifacts.joblib`.

**Run this cell to reset for a new A/B test.**

In [ ]:
import shutil

# Directories to delete (intermediate data)
intermediate_phases = [
    'phase0_diagnostics',
    'phase1_ingestion',
    'phase2_normalization',
    'phase3_expansive_search',
    'phase4_niching',
    'phase5_refinement',
]

print('Cleaning intermediate data...')
for phase_name in intermediate_phases:
    phase_dir = OUTPUT_ROOT / phase_name
    if phase_dir.exists():
        shutil.rmtree(phase_dir)
        print(f'  Deleted {phase_name}/')

# Recreate empty directories for next run
for phase in cfg['output']['phases']:
    (OUTPUT_ROOT / phase).mkdir(parents=True, exist_ok=True)

print('\nDone. Ready for next run.')
print(f'Preserved: {OUTPUT_ROOT}/phase6_champion/ and pipeline_artifacts.joblib')